# Multi-patient CHB-MIT study (leave-one-patient-out) - camera-ready extension
Downloads the chosen CHB-MIT patients straight from PhysioNet (no manual upload), extracts the same 11 features per channel
from 2-s windows (common 18-channel bipolar montage so all patients are comparable), then runs **leave-one-patient-out (LOPO)**
for the five techniques + temporal smoothing. Scaler/PCA are fitted on training patients only; smoothing parameters are chosen
by an inner leave-one-training-patient-out loop, never on the held-out patient.
Cells 2-4: LOPO on CHB-MIT; cells 5-6: Siena cross-dataset test with the same channels. Run the cells in order (Runtime > Run all). Default: 22 CHB-MIT cases and up to 2 seizure recordings per Siena subject. Total time: roughly 4-6 h on free Colab (it resumes from the Drive cache if disconnected, just run all again). For a quick test, shorten the PATIENTS list in cell 2. Features are cached, so a re-run is fast.

In [ ]:
!pip -q install mne imbalanced-learn xgboost
import os, re, gc, time, warnings, requests
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/epilepsy_multi'
except Exception:
    CACHE = '/content/epilepsy_multi'
os.makedirs(CACHE, exist_ok=True); os.makedirs('/content/edf', exist_ok=True)
print('cache folder:', CACHE)

In [ ]:
# ---------------- CONFIG ----------------
PATIENTS   = [f'chb{i:02d}' for i in range(1, 24) if i != 21]   # 22 distinct cases (chb21 is the same subject as chb01, so it is left out)
N_NONSEIZ  = 8        # non-seizure files kept per patient (all seizure files are always kept)
BASE = 'https://physionet.org/files/chb-mit-scalp-eeg-database/1.0.0'
TARGET = ['FP1-F7','F7-T7','T7-P7','P7-O1','FP1-F3','F3-C3','C3-P3','P3-O1',
          'FP2-F4','F4-C4','C4-P4','P4-O2','FP2-F8','F8-T8','T8-P8','P8-O2','FZ-CZ','CZ-PZ']   # 18 common channels

def http_get(url, path=None, tries=3):
    """Download with wget first (worked for PhysioNet from Colab before), then requests with a browser User-Agent."""
    import subprocess
    tmp = path or '/content/_tmp_dl.txt'
    last = ''
    for t in range(tries):
        r = subprocess.run(['wget', '-q', '-c', '--tries=3', '--timeout=60', '-U', 'Mozilla/5.0', '-O', tmp, url],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(tmp) and os.path.getsize(tmp) > 0:
            if path is None:
                txt = open(tmp, errors='ignore').read(); os.remove(tmp); return txt
            return path
        last = f'wget rc={r.returncode} {r.stderr[-200:]}'
        try:
            rq = requests.get(url, stream=True, timeout=120, headers={'User-Agent': 'Mozilla/5.0'})
            rq.raise_for_status()
            if path is None: return rq.text
            with open(path, 'wb') as f:
                for ch in rq.iter_content(1 << 20): f.write(ch)
            return path
        except Exception as e:
            last += f' | requests: {e}'
        print('  retry', t + 1, url.split('/')[-1], last[:300]); time.sleep(3)
    raise RuntimeError('download failed: ' + url + ' :: ' + last)

MIRRORS = {
  'chb':   ['https://physionet-open.s3.amazonaws.com/chbmit/1.0.0', 'https://storage.googleapis.com/chbmit-1.0.0.physionet.org',
            'https://physionet.org/files/chb-mit-scalp-eeg-database/1.0.0'],
  'siena': ['https://physionet-open.s3.amazonaws.com/siena-scalp-eeg/1.0.0', 'https://storage.googleapis.com/siena-scalp-eeg-1.0.0.physionet.org',
            'https://physionet.org/files/siena-scalp-eeg/1.0.0']}
def fetch(kind, rel, path=None):
    """try every mirror (AWS open-data, Google Cloud, PhysioNet) until one works"""
    errs = []
    for b in MIRRORS[kind]:
        try: return http_get(f'{b}/{rel}', path, tries=1)
        except Exception as e: errs.append(str(e)[-160:])
    raise RuntimeError(f'all mirrors failed for {rel}: ' + ' || '.join(errs))

print('--- mirror check ---')
for b in MIRRORS['chb']:
    try: http_get(f'{b}/chb01/chb01-summary.txt', tries=1); print('OK  ', b)
    except Exception as e: print('FAIL', b, str(e)[-150:])
print('--------------------')

def parse_summary(text):
    """file name -> list of (start_s, end_s); works for both summary formats (with/without seizure number)."""
    out = {}
    for blk in text.split('File Name:')[1:]:
        name = blk.split()[0]
        st = [int(x) for x in re.findall(r'Seizure\s*\d*\s*Start Time:\s*(\d+)', blk)]
        en = [int(x) for x in re.findall(r'Seizure\s*\d*\s*End Time:\s*(\d+)', blk)]
        out[name] = list(zip(st, en))
    return out

def pick_idx(names):
    up = [n.strip().upper() for n in names]; idx = []
    for t in TARGET:
        c = [i for i, n in enumerate(up) if n == t or n.startswith(t + '-')]
        if not c: return None
        idx.append(c[0])
    return idx

from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid
def window_features(data, sf, seiz):
    """data (n_ch, n_samples) -> features (n_win, n_ch*11) and labels (n_win,)"""
    w = int(2 * sf); n_win = data.shape[1] // w
    X = data[:, :n_win * w].reshape(data.shape[0], n_win, w).transpose(1, 0, 2).astype(np.float32)
    y = np.zeros(n_win, dtype=np.int8)
    for (s, e) in seiz:
        a = np.arange(n_win) * w
        y[(a + w > s * sf) & (a < e * sf)] = 1
    f = [X.mean(2), X.var(2), np.sqrt((X ** 2).mean(2)), skew(X, axis=2), kurtosis(X, axis=2), X.max(2), X.min(2)]
    freqs = np.fft.rfftfreq(w, d=1 / sf); P = (np.abs(np.fft.rfft(X, axis=2)) ** 2 / w).astype(np.float32)
    for lo, hi in [(0.5, 4), (4, 8), (8, 13), (13, 30)]:
        m = (freqs >= lo) & (freqs < hi)
        f.append(trapezoid(P[:, :, m], freqs[m], axis=2))
    F = np.stack(f, axis=2).reshape(n_win, -1).astype(np.float32)
    F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)   # flat/zero channels make skew/kurtosis NaN
    return F, y

import mne; mne.set_log_level('ERROR')
for pt in PATIENTS:
    cf = f'{CACHE}/{pt}.npz'
    if os.path.exists(cf): print(pt, 'cached'); continue
    print('==', pt)
    summ = parse_summary(fetch('chb', f'{pt}/{pt}-summary.txt'))
    seiz_files = [f for f, s in summ.items() if len(s) > 0]
    non = [f for f, s in summ.items() if len(s) == 0]
    keep = seiz_files + [non[i] for i in np.linspace(0, len(non) - 1, min(N_NONSEIZ, len(non))).astype(int)]
    FF, YY, ID = [], [], []
    for f in keep:
        p = f'/content/edf/{f}'
        try:
            fetch('chb', f'{pt}/{f}', p)
            raw = mne.io.read_raw_edf(p, preload=True, verbose=False)
            idx = pick_idx(raw.ch_names)
            if idx is None: print('  skip (channels missing):', f); os.remove(p); continue
            if abs(raw.info['sfreq'] - 256) > 1: print('  skip (sfreq):', f); os.remove(p); continue
            F, y_ = window_features(raw.get_data()[idx], raw.info['sfreq'], summ[f])
            FF.append(F); YY.append(y_); ID += [f] * len(y_)
            print(f'  {f}: {len(y_)} windows, {int(y_.sum())} seizure')
        except Exception as e:
            print('  FAILED', f, e)
        finally:
            if os.path.exists(p): os.remove(p)
            gc.collect()
    if not FF: print('  no usable files for', pt); continue
    np.savez_compressed(cf, X=np.concatenate(FF), y=np.concatenate(YY), fid=np.array(ID))

Xs, ys, pids, fids = [], [], [], []
PATIENTS = [p for p in PATIENTS if os.path.exists(f'{CACHE}/{p}.npz')]   # drop any patient with no usable files
for pt in PATIENTS:
    d = np.load(f'{CACHE}/{pt}.npz')
    Xs.append(d['X']); ys.append(d['y']); fids += list(d['fid']); pids += [pt] * len(d['y'])
Xf = np.nan_to_num(np.concatenate(Xs), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32); y = np.concatenate(ys); pid = np.array(pids); fid = np.array(fids)
del Xs; gc.collect()
print('\nTotal windows', Xf.shape, '| seizure windows', int(y.sum()), f'({100*y.mean():.2f}%)')
print(pd.DataFrame({'patient': PATIENTS,
                    'windows': [int((pid == p).sum()) for p in PATIENTS],
                    'seizure_windows': [int(y[pid == p].sum()) for p in PATIENTS]}).to_string(index=False))

In [ ]:
# ---------------- LOPO over all downloaded patients ----------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier
from scipy.ndimage import uniform_filter1d
from scipy.stats import wilcoxon, ttest_rel

RUN_TOMEK = False      # set False to save time (SMOTETomek was identical to SMOTE on chb05 earlier)
NEG_RATIO = 40   # training negatives are randomly undersampled to 40:1 (same for ALL techniques) so the run fits in free-Colab time; test patient is never subsampled
def RF(seed=42): return RandomForestClassifier(n_estimators=50, max_depth=10, random_state=seed, n_jobs=-1)
def sub(mask, seed):
    idx = np.where(mask)[0]; pos = idx[y[idx] == 1]; neg = idx[y[idx] == 0]
    neg = np.random.RandomState(seed).choice(neg, size=min(len(neg), NEG_RATIO * max(len(pos), 1)), replace=False)
    return np.sort(np.concatenate([pos, neg]))

def ev(name, yt, yp, pp):
    n = len(yt); fp = int(((yp == 1) & (yt == 0)).sum())
    return {'Model': name, 'P': precision_score(yt, yp, zero_division=0), 'R': recall_score(yt, yp, zero_division=0),
            'F1': f1_score(yt, yp, zero_division=0), 'AUC': roc_auc_score(yt, pp), 'FDR': fp / (n * 2 / 3600)}

def smooth(p, groups, k):
    if k <= 1: return p
    out = p.copy()
    for g in pd.unique(groups):
        ix = np.where(groups == g)[0]; out[ix] = uniform_filter1d(p[ix], size=k, mode='nearest')
    return out

KS, THS = [1, 3, 5, 7, 9, 15], [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60]
rows = []
test_pts = [p for p in PATIENTS if y[pid == p].sum() > 0]
for pt in test_pts:
    t0 = time.time()
    tr, te = pid != pt, pid == pt
    tri = sub(tr, 0)
    sc = StandardScaler().fit(Xf[tri]); pca = PCA(n_components=120, random_state=42).fit(sc.transform(Xf[tri]))
    Z = pca.transform(sc.transform(Xf)).astype(np.float32)
    Ztr, ytr, Zte, yte = Z[tri], y[tri], Z[te], y[te]
    ratio = (ytr == 0).sum() / max(int(ytr.sum()), 1)
    print(f'{pt}: test windows {len(yte)}, seizure {int(yte.sum())} | train ratio {ratio:.1f}:1', end=' ... ')
    Xsm, ysm = SMOTE(sampling_strategy=0.05, random_state=42).fit_resample(Ztr, ytr)
    models = {'Baseline RF': RF().fit(Ztr, ytr), 'SMOTE+RF': RF().fit(Xsm, ysm),
              'BalancedRF': BalancedRandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42).fit(Ztr, ytr),
              'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.08, scale_pos_weight=ratio, subsample=0.8, tree_method='hist',
                                       colsample_bytree=0.8, eval_metric='logloss', random_state=42, verbosity=0, n_jobs=-1).fit(Ztr, ytr)}
    if RUN_TOMEK:
        Xt, yt_ = SMOTETomek(smote=SMOTE(sampling_strategy=0.05, random_state=42), random_state=42, n_jobs=-1).fit_resample(Ztr, ytr)
        models['SMOTETomek+RF'] = RF().fit(Xt, yt_)
    else:
        models['SMOTETomek+RF'] = models['SMOTE+RF']
    for name, m in models.items():
        pp = m.predict_proba(Zte)[:, 1]; r = ev(name, yte, (pp >= 0.5).astype(int), pp); r['patient'] = pt; rows.append(r)
    # temporal smoothing + threshold: chosen on the TRAINING patients only (inner leave-one-training-patient-out)
    inner = []
    cand = [p for p in test_pts if p != pt]
    rng = np.random.RandomState(0); inner_pts = list(rng.choice(cand, size=min(4, len(cand)), replace=False))   # 5 inner folds keep 22-patient runtime manageable
    for q in inner_pts:
        itr = sub((pid != pt) & (pid != q), 1); iva = pid == q
        mm = RandomForestClassifier(n_estimators=20, max_depth=10, random_state=42, n_jobs=-1).fit(Z[itr], y[itr]); inner.append((mm.predict_proba(Z[iva])[:, 1], y[iva], fid[iva]))
    best, bf = (1, 0.5), -1
    for k in KS:
        for t in THS:
            f1s = [ev('', yy, (smooth(p_, g_, k) >= t).astype(int), smooth(p_, g_, k))['F1'] for p_, yy, g_ in inner]
            if np.mean(f1s) > bf: bf, best = np.mean(f1s), (k, t)
    k, t = best
    pp = smooth(models['Baseline RF'].predict_proba(Zte)[:, 1], fid[te], k)
    r = ev('Baseline RF + smoothing', yte, (pp >= t).astype(int), pp); r['patient'] = pt; rows.append(r)
    print(f'smoothing k={k}, thr={t}  [{time.time()-t0:.0f}s]')
    pd.DataFrame(rows).to_csv(f'{CACHE}/lopo_results.csv', index=False)   # checkpoint after every patient

df = pd.DataFrame(rows)[['patient', 'Model', 'P', 'R', 'F1', 'AUC', 'FDR']]
print('\n=== Per-patient results ==='); print(df.round(3).to_string(index=False))
agg = []
for name, g in df.groupby('Model', sort=False):
    d = {'Model': name, 'n_patients': len(g)}
    for kx in ['P', 'R', 'F1', 'AUC', 'FDR']:
        d[kx] = f'{g[kx].mean():.3f} +- {g[kx].std(ddof=0):.3f}'
    agg.append(d)
print('\n=== Mean +- std over held-out patients ==='); print(pd.DataFrame(agg).to_string(index=False))
print('\n=== Paired tests vs Baseline RF across patients (n = %d) ===' % len(test_pts))
base = df[df.Model == 'Baseline RF'].set_index('patient')
for name in df.Model.unique():
    if name == 'Baseline RF': continue
    o = df[df.Model == name].set_index('patient').loc[base.index]
    for kx in ['F1', 'AUC', 'FDR']:
        try: w = wilcoxon(base[kx], o[kx]).pvalue
        except ValueError: w = float('nan')
        print(f'{kx:3s} Baseline vs {name:24s} t-test p={ttest_rel(base[kx], o[kx]).pvalue:.4f}  Wilcoxon p={w:.4f}')
print('\nSend me ALL of this output. (Also saved to', CACHE + '/lopo_results.csv)')

In [ ]:
# ---------------- SIENA (cross-dataset test set) ----------------
# CHB-MIT uses bipolar channels; Siena is monopolar. We build the SAME 18 bipolar pairs from Siena (T3=T7, T4=T8, T5=P7, T6=P8),
# so channel i means the same thing in both datasets (the earlier version just took the first 23 Siena channels).
MAX_SIENA_PER_PT = 2     # seizure recordings used per Siena patient (raise to 2 for more data; each file is ~0.3-0.7 GB)
SIENA = 'https://physionet.org/files/siena-scalp-eeg/1.0.0'
SIENA_DRIVE = '/content/drive/MyDrive/siena/physionet.org/files/siena-scalp-eeg/1.0.0'   # used if the file is already there
PAIRS = [('FP1','F7'),('F7','T3'),('T3','T5'),('T5','O1'),('FP1','F3'),('F3','C3'),('C3','P3'),('P3','O1'),
         ('FP2','F4'),('F4','C4'),('C4','P4'),('P4','O2'),('FP2','F8'),('F8','T4'),('T4','T6'),('T6','O2'),('FZ','CZ'),('CZ','PZ')]

def parse_time(t):
    parts = t.strip().split('.')
    if len(parts) == 3:
        h, m, s = map(int, parts); return h * 3600 + m * 60 + s
    return None

def parse_siena(text):
    result = {}
    for block in re.split(r'Seizure n \d+', text)[1:]:
        fn = re.search(r'File name:\s*(\S+\.edf)', block)
        rs = re.search(r'Registration start time:\s*([\d.]+)', block)
        ss = re.search(r'Seizure start time:\s*([\d.]+)', block)
        se = re.search(r'Seizure end time:\s*([\d.]+)', block)
        if not all([fn, rs, ss, se]): continue
        reg, s1, s2 = parse_time(rs.group(1)), parse_time(ss.group(1)), parse_time(se.group(1))
        if None in (reg, s1, s2): continue
        on, off = s1 - reg, s2 - reg
        if on < 0: on += 86400
        if off < 0: off += 86400
        if on >= off: continue
        result.setdefault(fn.group(1), []).append((on, off))
    return result

def siena_bipolar(raw):
    up = {n.upper().replace('EEG', '').strip(): n for n in raw.ch_names}
    need = sorted({c for p in PAIRS for c in p})
    if any(c not in up for c in need): return None, [c for c in need if c not in up]
    raw.pick([up[c] for c in need]); raw.load_data(); raw.resample(256, verbose=False)
    order = {c: i for i, c in enumerate(need)}
    d = raw.get_data()
    return np.stack([d[order[a]] - d[order[b]] for a, b in PAIRS]).astype(np.float32), []

sf_ = f'{CACHE}/siena.npz'
if os.path.exists(sf_):
    print('Siena cached')
else:
    SF, SY, SFID, SPID = [], [], [], []
    for i in range(14):
        pt = f'PN{i:02d}'
        try:
            seiz = parse_siena(fetch('siena', f'{pt}/Seizures-list-{pt}.txt'))
        except Exception as e:
            print(pt, 'annotation failed', e); continue
        for f in list(seiz)[:MAX_SIENA_PER_PT]:
            p = f'{SIENA_DRIVE}/{pt}/{f}'
            use_local = os.path.exists(p) and os.path.getsize(p) > 5e6
            if not use_local:
                p = f'/content/edf/{f}'
                try: fetch('siena', f'{pt}/{f}', p)
                except Exception as e: print('  download failed', f, e); continue
            try:
                raw = mne.io.read_raw_edf(p, preload=False, verbose=False)
                data, miss = siena_bipolar(raw)
                if data is None: print('  skip', f, 'missing channels', miss); continue
                F, y_ = window_features(data, 256.0, seiz[f])
                SF.append(F); SY.append(y_); SFID += [f] * len(y_); SPID += [pt] * len(y_)
                print(f'  {f}: {len(y_)} windows, {int(y_.sum())} seizure')
            except Exception as e:
                print('  FAILED', f, e)
            finally:
                if not use_local and os.path.exists(p): os.remove(p)
                gc.collect()
    np.savez_compressed(sf_, X=np.concatenate(SF), y=np.concatenate(SY), fid=np.array(SFID), pid=np.array(SPID))
d = np.load(sf_)
Xsiena, ysiena, fsiena = np.nan_to_num(d['X'], nan=0.0, posinf=0.0, neginf=0.0), d['y'], d['fid']
print('Siena windows', Xsiena.shape, '| seizure windows', int(ysiena.sum()), '| patients', len(set(d['pid'])))

In [ ]:
# ---------------- CHB-MIT (all downloaded patients) -> Siena ----------------
sc = StandardScaler().fit(Xf); pca = PCA(n_components=120, random_state=42).fit(sc.transform(Xf))
Z = pca.transform(sc.transform(Xf)).astype(np.float32); ratio = (y == 0).sum() / max(int(y.sum()), 1)
sc_t = StandardScaler().fit(Xsiena)                       # target-domain standardisation (no labels used)
Zs = pca.transform(sc_t.transform(Xsiena)).astype(np.float32)
print(f'train ratio {ratio:.1f}:1 | variance explained by 120 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%')
Xsm, ysm = SMOTE(sampling_strategy=0.05, random_state=42).fit_resample(Z, y)
models = {'Baseline RF': RF().fit(Z, y), 'SMOTE+RF': RF().fit(Xsm, ysm),
          'BalancedRF': BalancedRandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42).fit(Z, y),
          'XGBoost': XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.05, scale_pos_weight=ratio, subsample=0.8,
                                   colsample_bytree=0.8, eval_metric='logloss', random_state=42, verbosity=0, n_jobs=-1).fit(Z, y)}
if RUN_TOMEK:
    Xt, yt_ = SMOTETomek(smote=SMOTE(sampling_strategy=0.05, random_state=42), random_state=42, n_jobs=-1).fit_resample(Z, y)
    models['SMOTETomek+RF'] = RF().fit(Xt, yt_)
else:
    models['SMOTETomek+RF'] = models['SMOTE+RF']
res = []
for name, m in models.items():
    pp = m.predict_proba(Zs)[:, 1]; res.append(ev(name, ysiena, (pp >= 0.5).astype(int), pp))
# smoothing + threshold chosen on the CHB-MIT patients only (inner leave-one-patient-out), then applied once to Siena
inner = []
for q in [p for p in PATIENTS if y[pid == p].sum() > 0][::2]:   # every 2nd patient keeps this step fast
    mm = RandomForestClassifier(n_estimators=40, max_depth=10, random_state=42, n_jobs=-1).fit(Z[pid != q], y[pid != q]); inner.append((mm.predict_proba(Z[pid == q])[:, 1], y[pid == q], fid[pid == q]))
best, bf = (1, 0.5), -1
for k in KS:
    for t in THS:
        f1s = [ev('', yy, (smooth(p_, g_, k) >= t).astype(int), smooth(p_, g_, k))['F1'] for p_, yy, g_ in inner]
        if np.mean(f1s) > bf: bf, best = np.mean(f1s), (k, t)
k, t = best; print(f'smoothing chosen on CHB-MIT: k={k}, thr={t}')
pp = smooth(models['Baseline RF'].predict_proba(Zs)[:, 1], fsiena, k)
res.append(ev('Baseline RF + smoothing', ysiena, (pp >= t).astype(int), pp))
dfs = pd.DataFrame(res)
print('\n=== CHB-MIT (all patients) -> Siena, default threshold 0.5 ==='); print(dfs.round(3).to_string(index=False))
print('\n=== ORACLE threshold sweep on Siena (upper bound, uses Siena labels; do not present as deployable) ===')
out = []
for name in ['Baseline RF', 'XGBoost']:
    p_ = models[name].predict_proba(Zs)[:, 1]
    for th in [0.5, 0.3, 0.2, 0.1, 0.05, 0.02]:
        r = ev(name, ysiena, (p_ >= th).astype(int), p_); r['thr'] = th; out.append(r)
print(pd.DataFrame(out)[['Model', 'thr', 'P', 'R', 'F1', 'FDR']].round(3).to_string(index=False))
dfs.to_csv(f'{CACHE}/siena_results.csv', index=False)
print('\nSend me ALL output of this cell and the previous one.')